# Laboratorio 1 — Series de Tiempo

## Análisis exploratorio general y serie total mensual

Este notebook contiene la preparación de datos, control de calidad, análisis exploratorio general y análisis preliminar de la serie obligatoria de viajeros internacionales.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

ROOT = Path.cwd()
INPUT = ROOT / 'data' / 'data.xlsx'
OUT = ROOT / 'resultados_laboratorio_1'
FIG = OUT / 'figuras'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 140

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIG / nombre, bbox_inches='tight')

def fmt(valor):
    return f'{valor:,.0f}'


ModuleNotFoundError: No module named 'matplotlib'

## 1. Carga, tipos de datos y calidad

Se eliminan únicamente duplicados exactos y registros sin fecha o cantidad válida. Los valores atípicos se identifican, pero no se eliminan automáticamente: pueden ser flujos reales y quitarlos distorsionaría los totales mensuales.

In [ ]:
datos = pd.read_excel(INPUT, sheet_name='Datos')
datos.columns = datos.columns.str.strip()
filas_iniciales = len(datos)

datos['Año'] = pd.to_numeric(datos['Año'], errors='coerce')
datos['Mes cod'] = pd.to_numeric(datos['Mes cod'], errors='coerce')
datos['Viajero'] = pd.to_numeric(datos['Viajero'], errors='coerce')
datos['fecha'] = pd.to_datetime(dict(year=datos['Año'], month=datos['Mes cod'], day=1), errors='coerce')

duplicados = int(datos.duplicated().sum())
datos = datos.drop_duplicates().copy()
validos = datos['fecha'].notna() & datos['Viajero'].notna() & datos['Viajero'].ge(0)
filas_excluidas = int((~validos).sum())
datos = datos.loc[validos].copy()

q1, q3 = datos['Viajero'].quantile([.25, .75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atipicos = int(((datos['Viajero'] < lim_inf) | (datos['Viajero'] > lim_sup)).sum())
faltantes = datos.drop(columns='fecha').isna().sum().sort_values(ascending=False)

calidad = pd.DataFrame({'Indicador': ['Filas originales', 'Duplicados exactos eliminados', 'Filas excluidas por fecha/cantidad inválida', 'Filas finales', 'Valores faltantes restantes', 'Atípicos IQR en registros', 'Límite superior IQR'],
                        'Valor': [filas_iniciales, duplicados, filas_excluidas, len(datos), int(faltantes.sum()), atipicos, lim_sup]})
display(calidad)
display(pd.DataFrame({'Variable': faltantes.index, 'Faltantes': faltantes.values, 'Porcentaje': (faltantes.values/len(datos)*100).round(2)}))
display(datos['Viajero'].describe(percentiles=[.25, .5, .75, .95, .99]).to_frame('Viajero'))

calidad.to_csv(OUT / 'calidad_datos.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'variable': faltantes.index, 'valores_faltantes': faltantes.values, 'porcentaje': (faltantes.values/len(datos)*100).round(2)}).to_csv(OUT / 'valores_faltantes.csv', index=False, encoding='utf-8-sig')
datos['Viajero'].describe(percentiles=[.25, .5, .75, .95, .99]).to_frame('Viajero').to_csv(OUT / 'estadisticas_viajero.csv', encoding='utf-8-sig')

## 2. Análisis exploratorio general

La serie `Turista + Excursionista` se presenta como comparación suplementaria, porque el enunciado advierte que `Viajero` cambia de definición entre 2022 y 2023. La serie total reportada se conserva como la serie obligatoria.

In [ ]:
total = datos.groupby('fecha')['Viajero'].sum().sort_index().asfreq('MS', fill_value=0)
comparable = (datos.loc[datos['Tipo de Viajero'].isin(['Turista', 'Excursionista'])].groupby('fecha')['Viajero'].sum().sort_index().reindex(total.index, fill_value=0))
series = pd.DataFrame({'total_reportado': total, 'turista_mas_excursionista': comparable})
anual = series.resample('YS').sum()
anual.index = anual.index.year
anual.index.name = 'año'
series.to_csv(OUT / 'series_totales_mensuales.csv', encoding='utf-8-sig')
anual.to_csv(OUT / 'resumen_anual.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(total, label='Total reportado', color='#1f77b4', linewidth=1.3)
ax.plot(comparable, label='Turista + Excursionista', color='#ef8a62', linewidth=1.1)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'), color='grey', alpha=.12, label='Período pandemia')
ax.set(title='Ingreso mensual de viajeros internacionales', xlabel='Fecha', ylabel='Viajeros')
ax.legend(ncol=3, fontsize=8)
guardar(fig, '01_serie_total_mensual.png')
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
anual.plot(kind='bar', ax=ax, color=['#1f77b4', '#ef8a62'])
ax.set(title='Viajeros anuales: total reportado y serie comparable', xlabel='Año', ylabel='Viajeros')
ax.tick_params(axis='x', rotation=45)
guardar(fig, '02_resumen_anual.png')
plt.show()

for columna, archivo, titulo in [('País', '03_top_paises.png', 'Diez países/mercados con mayor ingreso acumulado'), ('Región', '04_top_regiones.png', 'Diez regiones con mayor ingreso acumulado'), ('Vía', '05_vias_ingreso.png', 'Ingreso acumulado por vía')]:
    top = datos.groupby(columna)['Viajero'].sum().sort_values(ascending=False).head(10)
    top.to_frame('viajeros').to_csv(OUT / f"top_{columna.lower().replace(' ', '_')}.csv", encoding='utf-8-sig')
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=top.values, y=top.index.astype(str), ax=ax, color='#3b7ddd')
    ax.set(title=titulo, xlabel='Viajeros acumulados', ylabel=columna)
    guardar(fig, archivo)
    plt.show()

## 3. Análisis preliminar — Total mensual de viajeros internacionales

Se evalúan inicio, fin y frecuencia; comportamiento gráfico; descomposición; ACF y prueba Dickey-Fuller Aumentada (ADF).

In [ ]:
descomp = seasonal_decompose(total, model='additive', period=12, extrapolate_trend='freq')
fig = descomp.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Descomposición aditiva de la serie total mensual', y=1.02)
guardar(fig, '06_descomposicion_total.png')
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(total, lags=min(48, len(total)//2 - 1), ax=ax, zero=False)
ax.set_title('Autocorrelación (ACF) — total mensual')
guardar(fig, '07_acf_total.png')
plt.show()

adf_nivel = adfuller(total, autolag='AIC')
adf_diff = adfuller(total.diff().dropna(), autolag='AIC')
diagnostico = pd.DataFrame({'Métrica': ['Inicio', 'Fin', 'Frecuencia', 'Observaciones', 'Media mensual', 'Desviación estándar', 'Coeficiente de variación', 'ACF rezago 1', 'ACF rezago 12', 'ADF nivel: estadístico', 'ADF nivel: p-valor', 'ADF primera diferencia: estadístico', 'ADF primera diferencia: p-valor'],
                           'Valor': [total.index.min().strftime('%Y-%m'), total.index.max().strftime('%Y-%m'), 'Mensual', len(total), total.mean(), total.std(), total.std()/total.mean(), total.autocorr(1), total.autocorr(12), adf_nivel[0], adf_nivel[1], adf_diff[0], adf_diff[1]]})
display(diagnostico)
diagnostico.to_csv(OUT / 'diagnostico_serie_total.csv', index=False, encoding='utf-8-sig')

## 4. Interpretación preliminar

La decisión de diferenciación debe confirmarse más adelante con PACF, modelos candidatos y diagnósticos de residuos.

In [ ]:
top_pais = datos.groupby('País')['Viajero'].sum().sort_values(ascending=False)
top_region = datos.groupby('Región')['Viajero'].sum().sort_values(ascending=False)
top_via = datos.groupby('Vía')['Viajero'].sum().sort_values(ascending=False)
cambio_2023 = anual.loc[2023, 'total_reportado'] / anual.loc[2022, 'total_reportado'] - 1
caida_2020 = anual.loc[2020, 'total_reportado'] / anual.loc[2019, 'total_reportado'] - 1
conclusion_adf = 'se rechaza' if adf_nivel[1] < .05 else 'no se rechaza'

display(Markdown(f'''
### Hallazgos principales

- La serie total tiene **{len(total)} observaciones mensuales**, desde **{total.index.min():%Y-%m}** hasta **{total.index.max():%Y-%m}**.
- La caída anual entre 2019 y 2020 fue de **{caida_2020:.1%}**, lo que evidencia el impacto de la pandemia.
- Entre 2022 y 2023 el total reportado cambió **{cambio_2023:.1%}**. Este resultado debe interpretarse junto con el cambio de definición de `Viajero` indicado en el enunciado.
- El principal país/mercado es **{top_pais.index[0]}** ({top_pais.iloc[0] / top_pais.sum():.1%} del acumulado), la región líder es **{top_region.index[0]}** ({top_region.iloc[0] / top_region.sum():.1%}) y la vía dominante es **{top_via.index[0]}** ({top_via.iloc[0] / top_via.sum():.1%}).
- La ACF es alta en el rezago 1 ({total.autocorr(1):.3f}) y conserva asociación en el rezago 12 ({total.autocorr(12):.3f}), por lo que se evaluarán modelos estacionales.
- La ADF en nivel tiene p-valor **{adf_nivel[1]:.4f}**: al 5%, **{conclusion_adf}** la hipótesis nula de raíz unitaria. Como hay quiebres estructurales, se compararán modelos con `d=0` y, si los diagnósticos lo requieren, `d=1`; la diferenciación estacional de período 12 se probará solo si se justifica.
- Los registros atípicos no se eliminaron automáticamente, pues pueden corresponder a flujos reales agregados.
'''))

## 5. Análisis exploratorio — vías de ingreso y fronteras

Las tres vías se agregan por mes sin modificar el orden temporal. La participación se calcula sobre el total acumulado del período completo. Las fronteras se examinan dentro de cada vía para no mezclar puntos de ingreso de naturaleza distinta.

In [ ]:
vias_orden = ['Aérea', 'Terrestre', 'Marítima']
series_vias = (datos.groupby(['fecha', 'Vía'])['Viajero'].sum()
                 .unstack('Vía')
                 .reindex(total.index, fill_value=0)
                 .reindex(columns=vias_orden, fill_value=0))
series_vias.index.name = 'fecha'
series_vias.columns.name = None
series_vias.to_csv(OUT / 'series_mensuales_por_via.csv', encoding='utf-8-sig')

participacion_vias = series_vias.sum().sort_values(ascending=False).to_frame('viajeros_acumulados')
participacion_vias['participacion'] = participacion_vias['viajeros_acumulados'] / participacion_vias['viajeros_acumulados'].sum()
participacion_vias.to_csv(OUT / 'participacion_por_via.csv', encoding='utf-8-sig')
display(participacion_vias.assign(participacion=lambda x: x['participacion'].map('{:.2%}'.format)))

fronteras = (datos.groupby(['Vía', 'Frontera'])['Viajero'].sum()
             .rename('viajeros_acumulados').reset_index())
fronteras['participacion_en_via'] = fronteras['viajeros_acumulados'] / fronteras.groupby('Vía')['viajeros_acumulados'].transform('sum')
fronteras = fronteras.sort_values(['Vía', 'viajeros_acumulados'], ascending=[True, False])
fronteras.to_csv(OUT / 'fronteras_por_via.csv', index=False, encoding='utf-8-sig')
top_fronteras = fronteras.groupby('Vía', group_keys=False).head(5).copy()
display(top_fronteras.assign(participacion_en_via=lambda x: x['participacion_en_via'].map('{:.2%}'.format)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.45]})
orden_participacion = participacion_vias.sort_values('viajeros_acumulados', ascending=True)
axes[0].barh(orden_participacion.index, orden_participacion['participacion'], color=['#4c78a8', '#f58518', '#54a24b'])
axes[0].xaxis.set_major_formatter(PercentFormatter(1))
axes[0].set(title='Participación acumulada por vía', xlabel='Participación del total', ylabel='Vía')
sns.barplot(data=top_fronteras, x='viajeros_acumulados', y='Frontera', hue='Vía', dodge=False, ax=axes[1], palette='Set2')
axes[1].set(title='Cinco fronteras principales por vía', xlabel='Viajeros acumulados', ylabel='Frontera')
axes[1].legend(title='Vía', fontsize=8)
guardar(fig, '08_participacion_vias_y_fronteras.png')
plt.show()

fig, ax = plt.subplots(figsize=(13, 5))
for via, color in zip(vias_orden, ['#4c78a8', '#f58518', '#54a24b']):
    ax.plot(series_vias.index, series_vias[via], label=via, linewidth=1.25, color=color)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'), color='grey', alpha=.12, label='Período pandemia')
ax.set(title='Series mensuales de viajeros por vía de ingreso', xlabel='Fecha', ylabel='Viajeros')
ax.legend(ncol=4, fontsize=8)
guardar(fig, '09_series_mensuales_por_via.png')
plt.show()

## 6. Análisis preliminar — vía aérea

Para la serie aérea se evalúan cobertura temporal, descomposición, ACF, ADF y una decisión preliminar sobre transformación y diferenciación.

In [ ]:
aerea = series_vias['Aérea'].asfreq('MS', fill_value=0)
descomp_aerea = seasonal_decompose(aerea, model='additive', period=12, extrapolate_trend='freq')
fig = descomp_aerea.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Descomposición aditiva — vía aérea', y=1.02)
guardar(fig, '10_descomposicion_aerea.png')
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(aerea, lags=min(48, len(aerea)//2 - 1), ax=ax, zero=False)
ax.set_title('Autocorrelación (ACF) — vía aérea')
guardar(fig, '11_acf_aerea.png')
plt.show()

adf_aerea_nivel = adfuller(aerea, autolag='AIC')
adf_aerea_log = adfuller(np.log1p(aerea), autolag='AIC')
adf_aerea_diff = adfuller(aerea.diff().dropna(), autolag='AIC')
diagnostico_aerea = pd.DataFrame({
    'Métrica': ['Inicio', 'Fin', 'Frecuencia', 'Observaciones', 'Media mensual', 'Desviación estándar',
                'Coeficiente de variación', 'Mínimo mensual', 'Máximo mensual', 'ACF rezago 1', 'ACF rezago 12',
                'ADF nivel: estadístico', 'ADF nivel: p-valor', 'ADF log1p: p-valor',
                'ADF primera diferencia: estadístico', 'ADF primera diferencia: p-valor'],
    'Valor': [aerea.index.min().strftime('%Y-%m'), aerea.index.max().strftime('%Y-%m'), 'Mensual', len(aerea),
              aerea.mean(), aerea.std(), aerea.std()/aerea.mean(), aerea.min(), aerea.max(),
              aerea.autocorr(1), aerea.autocorr(12), adf_aerea_nivel[0], adf_aerea_nivel[1],
              adf_aerea_log[1], adf_aerea_diff[0], adf_aerea_diff[1]]
})
display(diagnostico_aerea)
diagnostico_aerea.to_csv(OUT / 'diagnostico_serie_aerea.csv', index=False, encoding='utf-8-sig')

p_adf_aerea = adf_aerea_nivel[1]
decision_d = 'd=0 como primer candidato' if p_adf_aerea < .05 else 'd=1 como primer candidato'
decision_transformacion = ('La escala original puede conservarse como referencia; se comparará log1p en los modelos para estabilizar la amplitud.'
                            if aerea.std()/aerea.mean() >= .30 else
                            'No hay evidencia descriptiva suficiente para exigir transformación; se conservará la escala original como referencia.')
top_aerea = fronteras.loc[fronteras['Vía'].eq('Aérea')].iloc[0]
display(Markdown(f'''
### Interpretación preliminar de la vía aérea

- La serie aérea contiene **{len(aerea)} observaciones mensuales**, de **{aerea.index.min():%Y-%m}** a **{aerea.index.max():%Y-%m}**, sin huecos tras la agregación mensual.
- La frontera aérea con mayor flujo acumulado es **{top_aerea['Frontera']}**, con **{top_aerea['participacion_en_via']:.1%}** de los viajeros que ingresan por esta vía.
- La descomposición permite separar una tendencia afectada por la pandemia y el cambio de definición posterior a 2022, de un patrón estacional mensual que se evaluará formalmente en el modelado.
- La ACF es {aerea.autocorr(1):.3f} en el rezago 1 y {aerea.autocorr(12):.3f} en el rezago 12; por ello se compararán especificaciones con componente estacional de período 12.
- ADF en nivel: p-valor **{p_adf_aerea:.4f}**; al 5%, {'se rechaza' if p_adf_aerea < .05 else 'no se rechaza'} la hipótesis nula de raíz unitaria. La decisión preliminar es probar **{decision_d}**, sin omitir alternativas por los quiebres estructurales.
- {decision_transformacion}
'''))